# Parallel Computing - Class 0 Diagnostic

Student ID: 230109011
Lecture group: 2-N
Practice group: 8-P


In [1]:
import time
import random
import math

## Task 1 - Trace the Code

In [2]:
x = 5
y = 2
print("start", x, y)

x = x + y   # 5 + 2 = 7
print(x, y)

y = x * 2   # 7 * 2 = 14
print(x, y)

x = y - x   # 14 - 7 = 7
print(x, y)

# final result
print("final:", x, y)

start 5 2
7 2
7 14
7 14
final: 7 14


## Task 2 - Find the Bug

`total = numbers[i]` just overwrites total each loop, doesn't add anything up.
So at the end total is just the last number in the list, 50.

Probably the code was supposed to sum the whole list. Fix is `+=` instead of `=`.

In [3]:
numbers = [10, 20, 30, 40, 50]

# original, buggy
total = 0
for i in range(len(numbers)):
    total = numbers[i]
print("buggy result:", total)

# fixed
total = 0
for i in range(len(numbers)):
    total += numbers[i]
print("fixed result:", total)

buggy result: 50
fixed result: 150


## Task 3 - Nested Loops

a) 10 * 10 = 100 prints
b) 1000 * 1000 = 1,000,000 prints
c) 10 * 1,000,000 = 10,000,000 prints

basically just outer * inner every time

In [4]:
print(10*10)
print(1000*1000)
print(10*1_000_000)

100
1000000
10000000


## Task 4 - Which Is Faster?

A only looks at the array once so it's O(n).
B compares everything with everything so it's O(n^2), way more work.

at n = 1,000,000 A does about a million steps but B does about 10^12, that's not even close.

In [5]:
n = 1_000_000
print("A steps approx:", n)
print("B steps approx:", n*n)

A steps approx: 1000000
B steps approx: 1000000000000


## Task 5 - Complexity Ranking

fastest growing -> slowest:

2^n, n^2, n log n, n, log n, 1

2^n and n^2 are the ones that get out of hand fast when n grows.

In [6]:
n = 30
print(1)
print(math.log2(n))
print(n)
print(n*math.log2(n))
print(n**2)
print(2**n)

1
4.906890595608519
30
147.20671786825557
900
1073741824


## Task 6 - Searching

If we don't know it's sorted -> just check every element one by one (linear search).
If we know it's sorted -> binary search, keep cutting the search space in half.

Binary is faster (log n vs n) but only works because the array is sorted.

In [7]:
def linear_search(arr, target):
    for i in range(len(arr)):
        if arr[i] == target:
            return i
    return -1


def binary_search(arr, target):
    lo = 0
    hi = len(arr)-1
    while lo <= hi:
        mid = (lo+hi)//2
        if arr[mid] == target:
            return mid
        if arr[mid] < target:
            lo = mid+1
        else:
            hi = mid-1
    return -1


arr = [3, 8, 12, 17, 24, 31, 45, 51, 63]

print(linear_search(arr, 45))
print(binary_search(arr, 45))

6
6


## Task 7 - Large Dataset

Just go through the numbers once and keep track of the biggest one so far.
Each number gets looked at exactly once, so O(n).

Don't need to keep all 500 million in memory at once, can just read them in chunks
(e.g. from disk) and update the running max as you go.

In [8]:
# fake version with small chunks just to show the idea
random.seed(1)

def get_chunks():
    for _ in range(5):
        yield [random.randint(0, 1000) for _ in range(1000)]

biggest = None
n_seen = 0
for chunk in get_chunks():
    for num in chunk:
        n_seen += 1
        if biggest is None or num > biggest:
            biggest = num

print(n_seen, biggest)

5000 1000


## Task 8 - Counting

use a dictionary, key = letter, value = count. simplest way to do this.

In [9]:
data = ["A","B","A","C","B","A","D","C","A","B"]

counts = {}
for x in data:
    if x in counts:
        counts[x] += 1
    else:
        counts[x] = 1

print(counts)

{'A': 4, 'B': 3, 'C': 2, 'D': 1}


## Task 9 - Matrix Operations

for two 2x2 matrices you need 8 multiplications total (2 per output cell, 4 cells).

In [10]:
A = [[1,2],[3,4]]
B = [[5,6],[7,8]]

# addition
add = [[A[i][j]+B[i][j] for j in range(2)] for i in range(2)]
print("A+B =", add)

# multiplication, done manually
result = [[0,0],[0,0]]
mults = 0
for i in range(2):
    for j in range(2):
        for k in range(2):
            result[i][j] += A[i][k]*B[k][j]
            mults += 1

print("A x B =", result)
print("mult ops:", mults)

A+B = [[6, 8], [10, 12]]
A x B = [[19, 22], [43, 50]]
mult ops: 8


## Task 10 - CPU vs Memory

Some reasons the loop could still be slow even with a fast CPU:

- memory bound - the CPU keeps waiting on data coming from RAM
- cache misses - array too big for the cache so it keeps hitting RAM
- no vectorization - loop runs one element at a time instead of using SIMD or multiple cores

## Task 11 - Running a Program

what happens roughly:

1. shell tells the OS to run ./program
2. OS creates a process, sets up memory for it
3. loader reads the file from disk into memory, loads any libraries it needs
4. OS jumps to main() and the CPU starts running instructions
5. if the program needs a file or network, it does a syscall and waits (I/O wait)
6. output goes to stdout, gets buffered, then shown on screen
7. when it's done the OS cleans up the memory/resources

## Task 12 - Four Programs

- A (lots of CPU) - CPU bound
- B (lots of memory) - memory bound
- C (reads big file) - I/O bound
- D (waits on user) - mostly idle

they can all run "at once" because the OS switches between them really fast (time slicing).

CPU time = how many cycles the scheduler gives the process + how much work it actually needs.

with only 1 core, all 4 have to take turns instead of running in parallel, so things slow down overall,
especially A since it can't just get its own core anymore.

## Task 13 - The Slow Program

not really, only if the whole thing is CPU bound. if some of the 10 hours is spent on disk/network
stuff that doesn't get faster just because the CPU is faster, so total speedup would be less than 2x.

this is basically Amdahl's law - you're limited by the part you can't speed up.

## Task 14 - Make It Faster

In [11]:
ideas = [
    "better algorithm (like n log n instead of n^2)",
    "fewer passes over data",
    "better data structure (dict/set instead of list)",
    "cache results instead of recomputing",
    "parallelize across cores",
    "vectorize with numpy",
    "less I/O, batch things",
    "better cache locality",
    "faster language / JIT",
    "avoid extra copies"
]
for i in ideas:
    print(i)

better algorithm (like n log n instead of n^2)
fewer passes over data
better data structure (dict/set instead of list)
cache results instead of recomputing
parallelize across cores
vectorize with numpy
less I/O, batch things
better cache locality
faster language / JIT
avoid extra copies


## Task 15 - Bottleneck

speeding up the small parts barely matters, C is the real bottleneck at 50s out of 59s total.

In [12]:
a,b,c,d,e = 2,2,50,3,2
before = a+b+c+d+e
print("before:", before)

after = a/10 + b/10 + c + d/10 + e/10
print("after:", after)
print("speedup:", before/after)

before: 59
after: 50.9
speedup: 1.1591355599214146


## Task 16 - 10 Million Marks

go through the data once, keep running min/max/sum/count for >=90 and <40.
for the median, use a counts array of size 101 (since marks are 0-100) instead of sorting
everything - way less memory and pretty much O(n) overall.

memory needed is basically just the 101-size array plus a few counters, not 10 million numbers.
probably I/O or memory bandwidth is the bottleneck since it's just reading through a lot of data.
could speed it up by splitting into chunks and running on multiple cores, then combining the results.

In [13]:
def analyze(marks):
    counts = [0]*101
    total = 0
    n = 0
    mn = None
    mx = None
    ge90 = 0
    lt40 = 0

    for m in marks:
        n += 1
        total += m
        counts[m] += 1
        if mn is None or m < mn:
            mn = m
        if mx is None or m > mx:
            mx = m
        if m >= 90:
            ge90 += 1
        if m < 40:
            lt40 += 1

    avg = total / n

    # find median from the counts array
    half = n // 2
    running = 0
    median = None
    for val in range(101):
        running += counts[val]
        if running > half:
            median = val
            break

    return n, mn, mx, avg, median, ge90, lt40


random.seed(2)
N = 200_000  # smaller number just for the demo, same algorithm works for 10 million

marks = [random.randint(0, 100) for _ in range(N)]

start = time.time()
n, mn, mx, avg, median, ge90, lt40 = analyze(marks)
elapsed = time.time() - start

print("count:", n)
print("min:", mn)
print("max:", mx)
print("avg:", avg)
print("median approx:", median)
print(">=90:", ge90)
print("<40:", lt40)
print("time:", elapsed)

count: 200000
min: 0
max: 100
avg: 50.08247
median approx: 50
>=90: 21796
<40: 79029
time: 0.02404189109802246
